In [ ]:
import csv
import json
import firebase_admin
from firebase_admin import credentials
from firebase_admin import firestore
import pandas as pd


In [ ]:
# Initialize Firebase Admin SDK
cred = credentials.Certificate("../DataCollector/serviceAccountKey.json")  # Replace with your own service account key
try:
    firebase_admin.initialize_app(cred)
except:
    pass

In [ ]:
# Initialize Firestore client
db = firestore.client()

fields = ['id','timestamp','email','latitude','longitude','gpsAccuracy','altitude','altitudeAccuracy','ssid','bssid','ipAddress','linkSpeed','connectionType','frequency','strength','txLinkSpeed','rxLinkSpeed']

def fetch_documents_to_csv(collection_name, csv_file):
    # Reference to the collection
    collection_ref = db.collection(collection_name)

    try:
        with open("last_doc.json","r") as f:
            last_timestamp = json.load(f)
    except:
        last_timestamp = 0
    # Query documents in ascending order of timestamp field'
    print("Querying Store...")
    docs = collection_ref.where("timestamp", ">", last_timestamp).order_by("timestamp").stream()

    print("Done")
    with open(csv_file, 'a', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fields)
        if csvfile.tell()==0:
            writer.writeheader()
        for doc in docs:
            # Write document to CSV
            d = {k:v for k,v in doc.to_dict().items()}
            d['id'] = doc.id
            writer.writerow(d)
            print(f"Wrote {d}")
            with open("last_doc.json","w") as f:
                json.dump(d["timestamp"],f)
    

fetch_documents_to_csv("Data", "Data.csv")


In [ ]:
df = pd.read_csv("Data.csv")
df.tail()

In [ ]:
df.drop_duplicates(inplace=True)
df.describe()

In [ ]:
pd.to_datetime(
    df['timestamp'],unit="ms"
    # ,utc=True
    )


In [ ]:
df.where(lambda x: x["gpsAccuracy"]<50).count()

In [ ]:
# # Initialize Firestore client
# db = firestore.client()

# def delete_collection(coll_ref, batch_size=500):
#     # docs = coll_ref.limit(batch_size).stream()
#     docs = coll_ref.where("timestamp", "<", 1713141088773).order_by("timestamp").stream()
#     deleted = 0
# # "1713141088773"
#     for doc in docs:
#         print(f'Deleting document {doc.id} => {doc.to_dict()}')
#         doc.reference.delete()
#         deleted += 1

#     if deleted >= batch_size:
#         return delete_collection(coll_ref, batch_size)

# # # Replace 'your-collection' with the name of your collection
# collection_ref = db.collection('Data')

# delete_collection(collection_ref)